# Análisis del equipo técnico

**PP2 · ITSE** — Inputs: `viviendas_procesadas.csv`, `tecnicos.csv`, `asignaciones.csv`, `visitas.csv`

Dos perspectivas: **jefe de área** (rendimiento comparativo, cobertura, alertas) y **técnico individual** (cola de prioridad, discrepancias con ONGs).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
DRIVE = Path('/content/drive/MyDrive/PP2')
Path('img').mkdir(exist_ok=True)

df_viv  = pd.read_csv(DRIVE / 'viviendas_procesadas.csv')
df_tec  = pd.read_csv(DRIVE / 'tecnicos.csv')
df_asig = pd.read_csv(DRIVE / 'asignaciones.csv')
df_vis  = pd.read_csv(DRIVE / 'visitas.csv')
col_avance = 'avance_obra' if 'avance_obra' in df_viv.columns else 'avanceObra'
print(f'Técnicos: {len(df_tec)} | Asignaciones: {len(df_asig)} | Visitas: {len(df_vis)}')

In [ ]:
# Carga y cobertura por técnico
# Regla de negocio del programa: máximo 2 visitas técnicas por obra (primera + segunda).
# El portal de ONGs cubre el avance intermedio con reportes fotográficos entre visitas.
# Por eso la métrica clave es cobertura_%: obras con al menos 1 visita / total asignadas.
# Un técnico con alta carga y baja cobertura está sobrecargado: tiene más obras de las que puede atender.
asig_c = df_asig.groupby('tecnico_id').size().reset_index(name='obras')
prim   = df_vis[df_vis['tipo']=='primera'].groupby('tecnico_id').size().reset_index(name='primeras')
seg    = df_vis[df_vis['tipo']=='segunda'].groupby('tecnico_id').size().reset_index(name='segundas')

carga = (df_tec[['id','apellido','nombre','departamentos']]
         .merge(asig_c, left_on='id', right_on='tecnico_id', how='left').drop(columns='tecnico_id')
         .merge(prim,   left_on='id', right_on='tecnico_id', how='left').drop(columns='tecnico_id')
         .merge(seg,    left_on='id', right_on='tecnico_id', how='left').drop(columns='tecnico_id')
         .fillna(0))
for c in ['obras','primeras','segundas']: carga[c] = carga[c].astype(int)
carga['sin_visita']  = carga['obras'] - carga['primeras']
carga['cobertura_%'] = (carga['primeras'] / carga['obras'].replace(0,1) * 100).round(1)
carga['nombre_c']    = carga['apellido'] + ', ' + carga['nombre']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].barh(carga['nombre_c'], carga['primeras'],   color='#22c55e', label='1ra visita', alpha=0.85)
axes[0].barh(carga['nombre_c'], carga['segundas'],   color='#3b82f6', label='2da visita',
             left=carga['primeras'], alpha=0.85)
axes[0].barh(carga['nombre_c'], carga['sin_visita'], color='#ef4444', label='Sin visitar',
             left=carga['primeras']+carga['segundas'], alpha=0.85)
axes[0].set_title('Cobertura de visitas por técnico'); axes[0].legend(fontsize=9)

col_p = ['#22c55e' if c>=75 else '#f59e0b' if c>=50 else '#ef4444' for c in carga['cobertura_%']]
axes[1].scatter(carga['obras'], carga['cobertura_%'], c=col_p, s=120, zorder=3)
for _, row in carga.iterrows():
    axes[1].annotate(row['apellido'], (row['obras'], row['cobertura_%']),
                     textcoords='offset points', xytext=(6,4), fontsize=8)
axes[1].axhline(70, color='#94a3b8', linestyle='--', alpha=0.6, label='70%')
axes[1].set_xlabel('Obras asignadas'); axes[1].set_ylabel('Cobertura (%)')
axes[1].set_title('Carga vs. cobertura'); axes[1].legend(fontsize=9)

plt.tight_layout(); plt.savefig('img/05_carga.png', bbox_inches='tight'); plt.show()
print(carga[['nombre_c','obras','primeras','segundas','sin_visita','cobertura_%']].to_string(index=False))

In [ ]:
# diferencia_ong = avance reportado por la ONG − avance verificado in situ por el técnico.
# Valores positivos (ONG sobreestimó) son el caso más preocupante:
# pueden indicar reporte inflado para justificar pagos parciales o evitar sanciones.
# Valores negativos (ONG subestimó) son menos críticos, la obra está mejor de lo declarado.
df_v = (df_vis
        .merge(df_viv[['num_exp','cuit_org',col_avance,'nivel_riesgo']],
               left_on='vivienda_id', right_on='num_exp', how='left')
        .merge(df_tec[['id','apellido']], left_on='tecnico_id', right_on='id', how='left'))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df_v['diferencia_ong'].dropna(), bins=25, color='#6366f1', edgecolor='white', alpha=0.85)
axes[0].axvline(0, color='#64748b', linestyle='--')
axes[0].axvline(df_v['diferencia_ong'].mean(), color='#ef4444', linestyle='--',
                label=f'Media: {df_v["diferencia_ong"].mean():.1f}')
axes[0].set_title('Distribución de discrepancias ONG vs. técnico')
axes[0].set_xlabel('Diferencia (puntos de avance — positivo = ONG sobreestimó)')
axes[0].legend(fontsize=9)

# Barra de media ± desvío por técnico: más directo que un boxplot para comparar niveles centrales
st_d = df_v.groupby('apellido')['diferencia_ong'].agg(['mean','std']).reset_index()
# Coloreado según la dirección de la discrepancia habitual de cada técnico
col_d = ['#ef4444' if m > 5 else '#f59e0b' if m > 0 else '#22c55e' for m in st_d['mean']]
axes[1].bar(st_d['apellido'], st_d['mean'], yerr=st_d['std'], capsize=5, color=col_d, alpha=0.85)
axes[1].axhline(0, color='#64748b', linestyle='--', alpha=0.6)
axes[1].set_title('Discrepancia promedio ± desvío por técnico\n(rojo = sobreestimación sistemática)')
axes[1].set_ylabel('Diferencia promedio (pp)')
plt.setp(axes[1].get_xticklabels(), rotation=30, ha='right')

plt.tight_layout(); plt.savefig('img/05_discrepancias.png', bbox_inches='tight'); plt.show()
print(df_v.groupby('apellido')['diferencia_ong'].mean().round(1).sort_values(ascending=False).to_string())

In [ ]:
# Obras activas sin ninguna visita
visitadas  = set(df_vis['vivienda_id'].unique())
activas_df = df_viv[df_viv['estado'].isin(['Iniciada','Avanzada'])].copy()
sin_visita = activas_df[~activas_df['num_exp'].isin(visitadas)]
urgentes   = sin_visita[sin_visita['nivel_riesgo']=='alto']

print(f'Obras activas sin visita: {len(sin_visita)} ({len(sin_visita)/len(activas_df)*100:.1f}%)')
print(f'De esas, en riesgo ALTO: {len(urgentes)}')
cols = ['num_exp','localidad','departamento',col_avance,'dias_activa','nivel_riesgo']
display(sin_visita[cols].sort_values('dias_activa', ascending=False).head(10))

## Score de priorización de visitas

El recurso más escaso del programa es la visita técnica (máximo 2 por obra). Con ~70% de las obras activas sin visitar, el técnico no puede ir a todas — necesita un **orden objetivo**.

Este score combina tres señales para responder *¿a qué obra voy primero?*:
- **Nivel de riesgo** de la obra (alto pesa más)
- **Tiempo expuesto** sin verificación (más días activa = más urgente)
- **Historial de la ONG** gestora (si suele sobreestimar, la obra necesita control antes)

El resultado es una lista ordenada lista para usar en terreno, sin depender del criterio subjetivo de cada técnico.

In [ ]:
# Score de priorización de visitas (obras activas sin ninguna visita)
C = {'base':'#4f46e5','ok':'#10b981','medio':'#f59e0b','alerta':'#f43f5e','neutro':'#94a3b8'}

# Sobreestimación promedio de cada ONG (positivo = reporta más de lo que el técnico verifica)
ong_sobre = (df_vis.merge(df_viv[['num_exp','cuit_org']], left_on='vivienda_id', right_on='num_exp')
                   .groupby('cuit_org')['diferencia_ong'].mean().clip(lower=0))

visitadas = set(df_vis['vivienda_id'])
sinv = df_viv[df_viv['estado'].isin(['Iniciada','Avanzada']) & ~df_viv['num_exp'].isin(visitadas)].copy()

# Tres componentes del score (suman ~0-100)
sinv['p_riesgo'] = sinv['nivel_riesgo'].map({'alto':50,'medio':30,'bajo':10})
sinv['p_tiempo'] = (sinv['dias_activa'] / sinv['dias_activa'].max() * 30).round()
sinv['p_ong']    = sinv['cuit_org'].map(ong_sobre).fillna(0).clip(0,20).round()
sinv['score']    = (sinv['p_riesgo'] + sinv['p_tiempo'] + sinv['p_ong']).round()
top = sinv.sort_values('score', ascending=False).head(15)

col_r = {'alto':C['alerta'],'medio':C['medio'],'bajo':C['ok']}
fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(top['num_exp'] + '  ' + top['localidad'].str[:14], top['score'],
        color=[col_r.get(r, C['neutro']) for r in top['nivel_riesgo']], alpha=0.9)
ax.invert_yaxis()
ax.set_title('Top 15 obras a visitar — score de prioridad\n(rojo = riesgo alto · ámbar = medio · verde = bajo)')
ax.set_xlabel('Score de prioridad')
plt.tight_layout(); plt.savefig('img/05_priorizacion.png', bbox_inches='tight'); plt.show()

print(f'Obras activas sin ninguna visita: {len(sinv)}')
display(top[['num_exp','localidad','departamento','nivel_riesgo','dias_activa',col_avance,'score']].head(10))

## Alerta de obras sin verificar (riesgo de sobre-reporte)

El caso más delicado: obras activas con **avance alto reportado pero cero visitas**. Nadie del ministerio confirmó ese avance — el número viene solo de la ONG. Si además la ONG gestora tiene historial de sobreestimar, la combinación es una bandera roja de sobre-reporte.

A diferencia del score de priorización (que ordena por urgencia operativa), esta vista apunta al **control y la transparencia**: son las obras donde el programa estaría pagando o certificando sobre datos no validados.

In [ ]:
# Alerta: obras activas sin visita con avance alto reportado
C = {'base':'#4f46e5','ok':'#10b981','medio':'#f59e0b','alerta':'#f43f5e','neutro':'#94a3b8'}

# Reutiliza 'sinv' y 'ong_sobre' de la celda anterior
sinv['ong_sobreestima'] = sinv['cuit_org'].map(ong_sobre).fillna(0) > 3
alerta = sinv[sinv[col_avance] >= 50].copy()

fig, ax = plt.subplots(figsize=(10, 5))
for flag, color, lab in [(False, C['base'], 'ONG con reporte fiable'),
                         (True,  C['alerta'], 'ONG que sobreestima')]:
    s = sinv[sinv['ong_sobreestima'] == flag]
    ax.scatter(s[col_avance], s['dias_activa'], c=color, alpha=0.55, s=28, label=lab)
ax.axvspan(50, 100, color=C['alerta'], alpha=0.06)           # zona de alerta
ax.axvline(50, color=C['alerta'], linestyle='--', alpha=0.6)
ax.set_xlabel('Avance reportado por la ONG (%)')
ax.set_ylabel('Días activa (sin verificación técnica)')
ax.set_title('Obras activas sin visita — zona de alerta: avance ≥50% sin verificar')
ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig('img/05_alerta.png', bbox_inches='tight'); plt.show()

print(f'Obras sin visita con avance ≥50% (alerta) : {len(alerta)}')
print(f'  de ellas, gestionadas por ONG que sobreestima: {alerta["ong_sobreestima"].sum()}')
display(alerta.sort_values(col_avance, ascending=False)[
    ['num_exp','localidad',col_avance,'dias_activa','cuit_org']].head(8))

In [ ]:
# Vista del técnico individual — cola de prioridad
TECNICO_ID = 3   # Ibáñez (sobrecargado) — cambiar para ver otro técnico

tec_info = df_tec[df_tec['id']==TECNICO_ID].iloc[0]
mis_ids  = df_asig[df_asig['tecnico_id']==TECNICO_ID]['vivienda_id']
mis_viv  = df_viv[df_viv['num_exp'].isin(mis_ids)].copy()

vis_map = df_vis[df_vis['tecnico_id']==TECNICO_ID].groupby('vivienda_id').size()
mis_viv['visitas'] = mis_viv['num_exp'].map(vis_map).fillna(0).astype(int)
mis_viv['pendiente'] = mis_viv.apply(
    lambda r: 'Sin visitar' if r['visitas']==0
    else ('Falta 2da' if r['visitas']==1 and r['estado'] in ['Iniciada','Avanzada']
    else 'Completo'), axis=1)

# Cola de prioridad: Sin visitar → Falta 2da → Completo.
# Dentro de cada grupo, riesgo alto primero (valor numérico más bajo = más urgente).
# Este orden garantiza que el técnico atiende primero lo más crítico
# sin depender de criterio subjetivo ni conocimiento previo de cada expediente.
mis_viv['_p'] = mis_viv['pendiente'].map({'Sin visitar':0,'Falta 2da':1,'Completo':2})
mis_viv['_r'] = mis_viv['nivel_riesgo'].map({'alto':0,'medio':1,'bajo':2})
mis_viv = mis_viv.sort_values(['_p','_r'])

print(f'Técnico: {tec_info["apellido"]}, {tec_info["nombre"]} | Zona: {tec_info["departamentos"]}')
print(f'Total asignadas: {len(mis_viv)}')
print(mis_viv['pendiente'].value_counts().to_string())

cols_v = ['num_exp','localidad','estado',col_avance,'nivel_riesgo','visitas','pendiente']
display(mis_viv[cols_v].head(15))